# Step 6 — Privacy Gate

We have medical imaging data, and before this data can be used by the rest of the platform, we need confidence that we are not accidentally exposing patient identity.


## 1. privacy checkpoint.
```
Raw data
   ↓
Privacy checks
   ↓
PASS? ─── NO → STOP
   │
  YES
   ↓
Create pseudonymized artifacts
   ↓
Continue with platform

 ```

## 2. What does "privacy" mean here?

Problem A — Is there identifying information?

For example:
- Patient names in metadata

Problem B — What do we do with patient IDs?

Our dataset has patient numbers such as:
- 049
Those numbers are not useful to the ML system.

But they are useful for identifying the patient within the original dataset.

So we replace them:
```
049
 ↓
PAT-98c32af848a5
```
That's pseudonymization.


## 3. Anonymization vs pseudonymization

This distinction is fundamental.

Anonymization
```
Patient 049
    ↓
remove identity
    ↓
cannot reconnect
```

There is no key that can bring the identity back.

Pseudonymization
```
Patient 049
    ↓
PAT-98c32af848a5
```
and privately:

and privately:

A protected key allows authorized people/system components to reconnect them.

So:
```
Anonymization
= identity destroyed

Pseudonymization
= identity replaced and protected
```


Here deliberately uses both concepts:
- the images were anonymized by the dataset authors
- our working patient identifiers are pseudonymized by MedImageForge

## Why not simply hash 049?

This is probably the most important technical decision in Step 6.

A beginner might think
```
049
 ↓
SHA-256
 ↓
abcdef123456...
```

Done.

But that's not secure enough here.

Why?

Because the possible patient IDs are tiny.

An attacker knows they are probably:
```
000
001
002
...
999
```
so they can calculate:
```
SHA256("000")
SHA256("001")
SHA256("002")
...
SHA256("999")
```
and fin the matching results

## Why HMAC?

Instead, the project uses:

**HMAC-SHA256**

which requires a secret key.

Conceptually:
```
patient ID + secret key
          ↓
      HMAC-SHA256
          ↓
     pseudonym
```
So:
```
049 + SECRET
      ↓
PAT-98c32af848a5
```
An attacker may know:
```
049
PAT-98c32af848a5
```
but does not know the secret.

Therefore they cannot simply reproduce the mapping.

## What is the secret actually doing?

The secret is essentially the key to the pseudonymization system.

The architecture is:
```
                 SECRET
                   │
                   │
Patient 049 ───────┼──→ HMAC ──→ PAT-98c32af848a5
                   │
                   │
              protected
```
The secret is stored separately from the source code:
```
environment variable
        OR
.secrets/pseudonym_salt

and .secrets/ is ignored by Git.
```
Why?

Because if you commit:

SECRET = abc123

to GitHub, you've defeated much of the protection.

## Why are pseudonyms deterministic?

Suppose patient 049 appears in three places:
```
labels.csv
demographics.csv
annotations
```
We want all three to refer to the same hidden patient.

Therefore:
```
049
 ↓
PAT-98c32af848a5
```
every time.

So:
```
labels
PAT-98c32af848a5
        │
        ├── demographics
        │
        └── annotations
```
This lets the platform join datasets without exposing the real patient ID.

That's why the pseudonym is deterministic.
